# Foundations of Artificial Neural Networks
## DS-3001: Foundations of Machine Learning

Content adapted from Terence Johnson (UVA)

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/03_Tuning_Testing_Evaluating'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous noteboook
os.chdir(path_to_DS_3001_folder)

**Defining a function to plot the weights of a neural network**

In [ ]:
# Networkx allows us to deal with graphs
import networkx as nx

def graph_layer_weights(W, layer):
    """ Function for plotting NN layers. """
    # Create graph
    G = nx.Graph()
    # Construct edges
    edges = []
    for i in range( (W[layer].shape)[0] ):
        for j in range( (W[layer].shape)[1] ):
            edges.append( ['i'+str(i),'h'+str(j), W[layer][i][j]] )
    # Add weighted edges
    G.add_weighted_edges_from(edges)
    # Position nodes using bipartite graph layout
    top = nx.bipartite.sets(G)[0]
    pos = nx.bipartite_layout(G,top)
    # Extract edge weights for visualization
    weights = nx.get_edge_attributes(G, 'weight')
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_size=10, node_color='red')
    # Draw edges with varying line width based on weight
    for edge, weight in weights.items():
        nx.draw_networkx_edges(G, pos, edgelist=[edge], width=weight * 5,alpha=.2)
    # Display plot
    plt.title("Neural Network Weights: Layer "+str(layer)+' to '+str(layer+1))
    plt.axis('off')
    plt.show()

## Artificial Neural Networks

- **How do we get more from our data?**
  - One of the challenges of our existing models is that we often struggle to "push" them to utilize the data more intensively

- **Intorducing the multi-layer perceptron (MLP):**
  - To be specific, we'll look at the **multi-layer perceptron**, or MLP. This is a pretty basic version of a neural network that works well for predictive analytics tasks like the kind we focus on

- **Use cases for artifical neural networks?**
  - You can use aNN for classification, regression, and unsupervised learning

- **When are ANN's best?**
  - If you have lots of feature-rich data with spatial/temporal correlation, ANN will work great
  - If you do not, the previous tools will work better

## From Logistic Regression to Neural Nets

- **What is a brain?**
  - Stimulus occurs (Input): You experience a sight/sound/smell/taste/touch
  - The stimulus causes a signal that reaches your brain
  - Neurons fire in response (Forward Pass) and you experience memories/thoughts/etc., or you don't even notice the signal was received
  - This process is noisy: Sometimes a smell like cinnamon catchs your attention and reminds you of a childhood memory; sometimes it doesn't

- **How do artifiical neural networks relate?**
  - The premise of an **artificial neural network** is to use this architecture of signal/response to build a machine intelligence
  - Artificial neural networks mimic the structure of the brain to mimic the function of human intelligence

- **What's the simplest version of a neural network?**
  - The entry-level model of a neuron is the Logistic Regression model (i.e. a multilayer perceptron with no hidden layers, just input and output)

# Classification with the Multilayer Perceptron

## Starting with a Single Layer Perceptron

- Let's start by writing out the model, then backtrack to talk over what it all means:
    1. **Input Layer**: $x = (x_1, ..., x_L)$
    > There is an input layer, $x$, that represents the data coming into the model.

    2. **Hidden layer**: $(z_1, ..., z_K)$.
    > There is a hidden layer comprised of $K$ **hidden nodes**

    3. **Activation function**: A(z), Ex. Logistic, ReLU, etc.
    > A non-linear activiation function is applied to each of the $K$ hidden nodes

    4. **Output layer** $\hat{y}$
    > An output layer that takes the output of the activation functions and aggregates it into a prediction

- **What design decisions do we need to make?**
  - The key choices here are the 1. number of hidden nodes and 2. the activation function
  - The input layer is determined by the data and the output layer is determined by the task (regression or classification)

## Starting from Logistic Regression

- **Can we put logistic regression in this framework?**
    - Let the activation function be $a(z) = 1/(1+e^{-z})$. Then logistic regression is:

\begin{alignat*}{2}
\hat{y}(x)_{LR} &=& a\left( b + \sum_{\ell=1}^L w_\ell x_\ell \right) \\
&=& a\left( b + w \cdot x \right)
\end{alignat*}

- Putting this into terms of logistic regression:
    1. An input layer, $x$
    2. No hidden layers
    3. The output layer, $a(z)$, with $z = b + w \cdot x$

<img src="https://github.com/ds4e/aNN/blob/main/src/zeroLayer.png?raw=true" width="520">

## Adding Hidden Layers

- A neural net with one hidden layer is slightly more general:
\begin{alignat*}{2}
z^0(x) &=&   \underbrace{a}_{\text{Activation Function}} \left( \underbrace{b^0}_{\text{Bias}} + \underbrace{\left[ \begin{array}{cccc} w_{11}^0 & w_{12}^0 & \dots & w_{1L}^0 \\ w_{21}^0 & w_{22}^0 & \dots & w_{2L}^0 \\ \vdots & \vdots & \ddots & \vdots \\ w_{K1}^0 & w_{K2}^0 & \dots & w_{KL}^0 \end{array} \right]}_{\text{Weights}} \underbrace{\left[ \begin{array}{c} x_1 \\ x_2 \\ \vdots \\ x_L \end{array}\right] }_{\text{Input Layer}}\right) \\
&=& a( b^0 + W^0 \cdot x)
\end{alignat*}

- Then you add the output layer, which looks like logistic regression on the hidden layer $z^0(x)$:
\begin{alignat*}{2}
\hat{y}(x)_{NN} &=&   \underbrace{a}_{\text{Activation Function}} \left( \underbrace{b^1}_{\text{Bias}} + \underbrace{\left[ \begin{array}{cccc} w_{1}^1 & w_{2}^1  & \cdots  & w_{K}^1 \end{array} \right]}_{\text{Weights}} \underbrace{\left[ \begin{array}{c} z_1^0(x) \\ z_2^0(x) \\ \vdots \\ z_K^0(x) \end{array}\right] }_{\text{Hidden Layer}}\right) \\
&=&  a \left(b^1 + w^1 \cdot z^0(x) \right)
\end{alignat*}

- **How can we write an ANN as an equation?**
  - Plunking the hidden layer into the output layer yields:
  $$
  \hat{y}(x)_{NN} = a \left(b^1 + w^1 \cdot a( b^0 + W^0 \cdot x)  \right)
  $$

- **What does this boil down to?**
  - This is essentially **Logistic Regression on the $K$ hidden nodes**, instead of the input layer

- **What are our hyperparameters?**
  - The hyperparameters here are really $K$ and the activation function $a$, since we'll choose $(W^0, b^0, w^1, b^1)$ through standard training by minimizing binary cross entropy

<img src="https://github.com/ds4e/aNN/blob/main/src/multipleLayer.png?raw=true" width="520">

## Neural Net Intuition

- Notice how this is logistic regression on the hidden layer, instead of the input layer

- **Here is how people think of neural networks**:
  - The weights/bias of the hidden layer represent the machine automatically doing feature engineering and learning about the non-linear relationships between the variables

- So instead of interacting variables and taking powers and so on by hand, **the hidden layer is exploring those relationships automatically and building the feature space for you**

- **Increasing depth increases model complexity:**
  - By adding more depth, you allow the model to investigate more complex relationships and interrelationships among the input layer variables, representing the machine gaining an "intuition" about the environment

- **Then the output layer is just a conventional model**:
  - Linear regression or logistic regression on the final hidden layer

## Neural Net Prediction
- So while this is certainly more complex than Linear or Logistic Regression, prediction is straightforward once we have weights:
$$
\hat{y}(x)_{NN} = a \left( \hat{b}^1 + \hat{w}^1 \cdot a( \hat{b}^0 + \hat{W}^0 \cdot x)  \right)
$$

- **Finding the solution is more complex:**
  - Getting the optimal biases and weights will obviously be rather complex: This is a much harder problem than solving the normal equations or doing gradient descent with logistic regression

# Classification Example: Heart Failure

* This is the same data example we looked at in the Logistic Regression model.

In [ ]:
# pd.set_option('display.max_columns', None)
cardiac_df = pd.read_csv('./data/heart_failure_clinical_records_dataset.csv')
print(cardiac_df.head())

# Isolate the outcome variable and the design matrix
y = cardiac_df['DEATH_EVENT']
X = cardiac_df.drop(['DEATH_EVENT'],axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

# Define a min max function
def MinMax(X):
  result = (X - X.min()) / (X.max() - X.min())
  return result

# Min max our features
X = X.apply(MinMax)

# Gather a single train test spilt for now
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=100
)


#### Fitting a MLP in Sklearn:

* We can fit an MLP Classifier using sklearn with the following import:
  - `from sklearn.neural_network import MLPClassifier`

* The documentation for the network is found here: [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)

* Once you fit the model, you can access the weights and intercepts of your `model` object as:
  - **Weights of the model:** `model.coefs_`
  - **Intercepts of the model:** `mode.intercepts_`

* Note: In practice, you won't be using Sklearn for fitting neural networks. You should instead use PyTorch or Tensorflow for fitting neural networks. Using Sklearn here will allow us to build intuition on what is actually happening.


In [ ]:
# The imports we need to fit a basic neural network
from sklearn.neural_network import MLPClassifier

## Fit neural network:
mlp_class = MLPClassifier(
    solver = 'adam', # What gradient descent solver do you want to use?
    hidden_layer_sizes=(12), # How many nodes should there be in each hidden layer?
    activation='logistic', # What activation function do you want to use?
    max_iter = 2000, # How many iterations should gradient descent be ran?
    verbose=False # Whether you want the progress to be described?
)
mlp_class = mlp_class.fit(X_train,y_train)

print('Accuracy: ', mlp_class.score(X_test, y_test))

In [ ]:
# Weights and biases:
W = mlp_class.coefs_
b = mlp_class.intercepts_

**Question:** What type of objects are W and b? What do they represent?

In [ ]:
# Check the types of w and b


In [ ]:
# What's inside w and b?


**Let's write out our equation for the neural network by hand in Python:**

In [ ]:

# Defining an activation function
def a(z):
    '''
    Defining an activation function
    Using the Logistic/Sigmoid activation function

    Inputs:
    -------
      z: Our latent variable

    Returns:
    -------
      y: The output from the activation function
    '''

    y = 1/(1+np.exp(-z))

    return y

# We can calculate the pieces of the neural network
# First, our latent variables in the hidden layer
z0 = a( b[0].T + X_test @ W[0] )

# Next, our
p_hat = a( b[1] + z0 @ W[1]) # Output layer

In [ ]:
# Comapre the predicted probabilities we calculated with what the model returns to us to verify
predicted_probs = mlp_class.predict_proba(X_test)[:, 1]

# They are the same!
p_hat.values.flatten() == predicted_probs

**Was it better to use a neural network in this case or does logistic regression perform better?**

In [ ]:
from sklearn.linear_model import LogisticRegression

# Fit logistic regression:
LR = LogisticRegression(
    penalty=None, # Remember to set penalty to None so that we don't accidentally use the L2 penalty
    solver='lbfgs', # The solver, based on documentation Adam is not available here
    max_iter=500 # The number of iterations to run thesolver
)
LR.fit(X_train,y_train)

# Calculate the accuracy
print('Accuracy: ', LR.score(X_test,y_test))


# Calculate our latent variables
lr_latent = LR.intercept_.T +  X_test @ LR.coef_.T

# Calculate the probabilies
p_hat_LR = a(lr_latent)

- The logistic regression model is performing just as well with less iterations and less learned coefficients.
- Neural networks typically need immense amounts of training data (rows and columns) to beat standard predictive analytics models, so this isn't a surprise

In [ ]:
# Comapre the number of weights and biases in the LR vs neural network

print('Number of Coefficients in Neural Network:', W[0].size + b[0].size + W[1].size + b[1].size)
print('Number of coefficients in Logistic Regression:', LR.intercept_.size + LR.coef_.size)

**Question:** Which would you choose?

**Let's graph what the weights of our model look like:**

In [ ]:
# Isolating the weigths and biases
W = mlp_class.coefs_
b = mlp_class.intercepts_

# Graphing the results from the input to the hidden layer
graph_layer_weights(W,0)

In [ ]:
# What about the hidden layer to the output layer?
graph_layer_weights(W,1)

In [ ]:
# We can plot the distribution of the values in the hidden layers
z = a(b[0].T + X_test @ W[0])
sns.kdeplot(z)
plt.title('Hidden Nodes')
plt.show()


In [ ]:
# We can plot out the latent variable for the output node
s = b[1] + z@W[1]
sns.kdeplot(s)
plt.title("Latent Variable")
plt.show()

In [ ]:
# Lastly, we can plot out the actual predicted probabilities
p_hat_NN = a(s)
sns.histplot(p_hat_NN)
plt.title('Predicted Probabilities of the Neural Net')
plt.show()

In [ ]:
# We can also look at the predicted probabilities from the logistic regression
sns.histplot(p_hat_LR)
plt.title('Predicted Probabilities of the Logistic Regression')
plt.show()

In [ ]:
# Create collects the predicted probabilites for the MLP and LR
gdf = pd.DataFrame(
    {
        'p_NN': p_hat[0].to_numpy(),
        'p_LR': p_hat_LR[0].to_numpy()
    }
)

sns.scatterplot(data = gdf, x='p_NN', y = 'p_LR')
plt.title('Predicted Probabilities\nLogistic Reg vs Neural Network')

# Plotting a 45 degree line to represent when the predicted probabilities are the same
plt.plot((0,1),(0,1),'k--')

plt.show()

## How Many Hidden Nodes should you include in your Hidden Layer?

- We can use the train/test split:
    1. Train the network on the training data and evaluate its performance on the test data, for a range of values for $K$ (the number of hidden nodes)

    2. Pick the value that maximizes accuracy on the test set

- There is not a firm mathematical answer to this question. We could also try cross validating.

In [ ]:
# Maximum number of hidden nodes to consider
K_max = 100

# Accuracy on test set
acc_test = []

# Accuracy on training set
acc_train = []

# Loop over potential values of k
for k in np.arange(1,K_max+1,3):

    # Fit neural network
    mlp_class_k = MLPClassifier(
        solver = 'adam',
        hidden_layer_sizes=(k),
        activation='logistic',
        max_iter = 2000
      )

    mlp_class_k = mlp_class_k.fit(X_train,y_train)
    acc_test.append( mlp_class_k.score(X_test, y_test) )
    acc_train.append( mlp_class_k.score(X_train, y_train) )

In [ ]:
sns.lineplot(x=  np.arange(1,K_max+1,3), y = acc_test, label='Test Accuracy')
sns.lineplot(x= np.arange(1,K_max+1,3),y=acc_train, label='Training Accuracy')
plt.show()

In [ ]:
reg = MLPClassifier(solver = 'adam',
                        hidden_layer_sizes=(18),
                        activation='logistic',
                        max_iter = 2000)
reg = reg.fit(X_train,y_train)
print( 'Accuracy: ', reg.score(X_test,y_test) )


In [ ]:
W = reg.coefs_
b = reg.intercepts_
graph_layer_weights(W,0)

In [ ]:
graph_layer_weights(W,1)

## Adding more layers

- So we can understand deep neural networks **recursively**:
    1. There is an input layer, $x$
    2. The first layer outputs $z^1 = a(b^0 + W^0 \cdot x)$
    3. Each subsequent layer generates more hidden node values, $z^k = a(b^{k-1} + W^{k-1} \cdot z^{k-1})$
    4. The output layer is the activation function applied to the final hidden layer, $a(b^K + w^K \cdot z^K)$
    
- It's alternating linear algebra with non-linear activation functions:
$$
\hat{y}_{NN}(x) = a(b^K + w^K \cdot a( b^{K-1} + W^{K-1} \cdot a( ... a(b^0 + W^0 \cdot x))))
$$

In [ ]:
from sklearn.neural_network import MLPClassifier

reg = MLPClassifier(
    solver = 'adam',
    hidden_layer_sizes=(25,7), # Having two hiden layers, one with 25 nodes, another with 7
    max_iter = 2000,
    activation='relu'
)

reg.fit(X_train,y_train)
print('Accuracy: ', reg.score(X_test,y_test))

In [ ]:
W = reg.coefs_
b = reg.intercepts_
graph_layer_weights(W,0)

In [ ]:
graph_layer_weights(W,1)

In [ ]:
graph_layer_weights(W,2)

**Question:** Fit a neural network with 4 hidden layers that each have 28 hidden nodes.

In [ ]:
# Answer Here


# Architecture Questions

## Key Choices

OK, so, you're ready to go. Every package will ask you to specify at least four things:

- 1. What **loss function**? Usually SSE for regression, cross entropy for classification
- 2. What **activation function**? The logistic/sigmoid is a classic, but ReLU is popular
- 3. How many **hidden layers** and **hidden nodes**?
- 4. Which **optimizer**? Stochastic Gradient Descent is popular, but there are other options

## 1. Common Loss Functions
- **Binary cross-entropy/log-loss**,
  - `binary_crossentropy`, for 0/1 classification: Letting $\hat{p}_i$ be the predicted probability that observation $y_i$ takes the value 1,
$$
L_{bce} = - \dfrac{1}{N} \sum_{i=1}^N y_i \log \left( \hat{p}_i \right) + (1-y_i) \log \left( 1 - \hat{p}_i\right)
$$
- **Categorical cross-entropy/log-loss**
  - `categorical_crossentropy`, for multi-label classification: Letting $\hat{p}_{ik}$ be the predicted probability that observation $i$ is in category $k$,
$$
L_{cce} = - \dfrac{1}{N} \sum_{i=1}^N \sum_{k=1}^K y_{ik} \log \left( \hat{p}_{ik} \right)
$$
- **Mean-squared error**
  - For regression: Letting $\hat{y}_i$ be the predicted value for observation $i$,
$$
L_{mse} = \dfrac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2
$$

## 2. Activation Functions

- The logistic function is a nice place to start because it ties together our linear models discussions with the story about neural networks mimicing human brains, but people often use other models of neurons.

- Just to show you what they look like, standards include:
    - **Rectified linear unit** or ReLU:
      $$a(z) = \max ( z, 0 ) $$
    - **Sigmoid** or **Logistic**:
      $$a(z) = 1/(1+\exp(-z))$$
    - **Leaky ReLU**:
      $$a(z) = \max\(.1z, z\)$$
    - **Tanh**:
      \begin{gather}
      a(z) = \dfrac{e^z-e^{-z}}{e^z+e^{-z}}
      \end{gather}

- **Ultimately, this is all similar to a linear model**:
  - We have to pick all these weights, so we'll be using the derivative/gradient to search for weights that provide the best fit

- **Why switch to something else?**
  - For every large values of the latent variable, the sigmoid/logistic function flattens out, and this can be inconvenient for training models

- The popular choice right now is ReLU, $a(z) = \max \{z, 0 \}$
- Why?
    - It is non-linear, so that it works as an activation function, but it is simple to compute
    - Unlike the logistic distribution, the slope never goes to zero ("vanishing gradients" problem)

In [ ]:
# Pick some Z's to look at
Z = np.linspace(-5.0,5.0,100)

# Plot the ReLU function regression
Y_logistic = np.exp(Z) / (1 + np.exp(Z))
sns.lineplot(x=Z,y=Y_logistic)
plt.title('Logistic / Sigmoid Activation')
plt.xlabel('z')
plt.ylabel('A(Z)')
plt.show()

# Plot the ReLU function regression
Y_relu = np.maximum(Z,0)
sns.lineplot(x=Z,y=Y_relu)
plt.title('ReLU Activation')
plt.xlabel('z')
plt.ylabel('A(Z)')
plt.show()

# Plot the Leaky ReLU function regression
Y_relu_leaky = np.where(Z > 0, Z, 0.1*Z)
sns.lineplot(x=Z,y=Y_relu_leaky)
plt.title('Leaky ReLU Activation')
plt.xlabel('z')
plt.ylabel('A(Z)')
plt.show()

# Plot the ReLU function regression
Y_tanh = np.tanh(Z)
sns.lineplot(x=Z,y=Y_tanh)
plt.title('TanH Activation')
plt.xlabel('z')
plt.ylabel('A(Z)')
plt.show()

## 3. Hidden Layers/Nodes
- **How many hidden layers?**
  - It depends
  - For the kinds of straightforward prediction problems we've looked at in class, probably not more than two; there are analyses and rules-of-thumb that people have developed that suggest two is fine for prediction-type tasks

- A popular aNN in computer vision (convolutional neural networks) has three layers, as a matter of mathematical necessity (a CNN must have 3 layers to work)
- More than three layers qualifies as "**deep learning**," which might surprise you

- The power of generative artificial intelligence and **large language models** comes from having a **very deep neural network**: An earlier version of ChatGPT had 96 hidden layers and 1.75 billion parameters, which cost about 100k to fit (and about 700k per day to run for users)

- Ex. Llama 3.3, which has 70 billion parameters

## 4. Optimizers
Somewhat in order of popularity:
- `adam`: Variation on stochastic gradient descent.
- `sgd`: Stochastic Gradient Descent. Does gradient descent in batches, randomly splitting the sample to avoid using too much data at once.
- `lbfgs`: Limited-memory Broyden Fletcher Goldfarb Shanno. Classical method. Approximates the optimal learning rate using just gradient calculations.

## Using SciKit-Learn
- Always maxmin normalize your input variables
- Supervised:
    - Regression: `sklearn.neural_network.MLPRegressor(activation='relu', hidden_layers=(100,), solver='adam', max_iter=200, random_state=None)`
    - Classification: `sklearn.neural_network.MLPClassifier(activation, hidden_layers=(100,), solver='adam', max_iter=200, random_state=None)`
- Unsupervised:
    - Restricted Boltzman Machine: `sklearn.neural_network.BernoulliRBM(n_components=256,learning_rate=0.1,n_iter=10,random_state=None)`
- You pass a tuple of layer sizes to `hidden_layers`, simultaneously determining the number of hidden layers as well

## Other Packages for aNN Analysis
- There are a variety of packages for neural network analysis (from least to most complex):
    - SciKit: No GPU support, but specifying layer/node architecture is just a tuple
    - Keras/Tensorflow: GPU support, Keras is a SciKit-style API to TensorFlow so it's reasonably easy to get started.
    - PyTorch: GPU support, very flexible, but even simple neural networks require a dozen lines of boilerplate code
- The GPU-accelerated aNN support is probably the main reason to use Python, not Sci-Kit
- PyTorch/TensorFlow also give you complete control over the network architecture: Using different activation functions in different layers, not dense connections from layer to layer, etc.
- In general: PyTorch is more innovative and flexible, TensorFlow is more stable and useful for deployment

## Common Misconceptions
- More nodes and layers **does not necessarily mean better results**
- The number of nodes in each hidden layers shouldn't necessarily decrease from input to output: **Some architectures shrink down and then expand** (e.g. variational auto encoder)
- **Neural networks aren't guaranteed to do better** than, say, a random forest or the LASSO --- in a lot of situations they have be very hard to fit

## ANNs as "Black Box" Models
- Linear regression is easy to interpret:
- Decision trees are easy to explain:
- Neural networks are neither